# BAGEL — DistilBERT Router vs Random Selection (9-model ensemble, source-disjoint test set)

Extends the prior notebook's `BAGEL_test_evaluation.ipynb` with a DistilBERT-routed curve
alongside its existing Random Selection curve, on the exact same 9-promptcop
ensemble, held-out test set, and threshold.

**Matches the prior notebook's setup exactly, not the earlier 9-original-dataset pipeline:**
- Ensemble: `model_1..model_8, model_10` (model_9/guychuk dropped, model_10
  trained on nvidia/Aegis replaces it — see `BAGEL_new_finetune.ipynb`)
- Threshold: fixed at **0.48** (its calibrated value), no per-router recalibration
- Held-out test set: its 5-dataset concat (hlyn-labs, allenai/wildjailbreak,
  LLM Past Tense, Necent, TrustAIRLab/JailbreakQR) — reused verbatim
- Selection mechanism: 1 routed pick + (n-1) random peers, same as the paper
  and same as this project's earlier work — Random Selection uses no routed
  pick (all n random), matching that notebook's code exactly (`random.seed(42)`,
  `random.sample`, same loop structure) so that curve is bit-for-bit
  reproducible against it.

**Router training (C_global):** the 10% calibration slice from each of the 8
original datasets (unchanged), plus a slice from Aegis sized to 10% of
Aegis's *total* row count, carved from the 80% `train_df` split
(`BAGEL_new_finetune.ipynb`, same `random_state=42`). The router is trained
on provenance labels (prompt text -> source dataset), never on model_10's
predictions, so model_10 having already seen those exact rows during its own
fine-tuning does not leak into the router's training signal.

## 0. Setup

In [ ]:
!pip -q install --upgrade huggingface-hub datasets transformers

In [ ]:
import os, gc, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
BASE_DIRECTORY     = ""   # same Drive folder as everything else -- model_10 lives here too
MODELS_DIRECTORY   = BASE_DIRECTORY + "models/"
DATASETS_DIRECTORY = BASE_DIRECTORY + "Datasets/"
RESULTS_DIRECTORY  = BASE_DIRECTORY + "ResultsTest/"
ROUTER_DIRECTORY   = RESULTS_DIRECTORY + "router/"   # same as the original notebook -- overwrites its C_global.csv/P_cal.npy/etc, confirmed fine

os.makedirs(ROUTER_DIRECTORY, exist_ok=True)

# --- knobs -------------------------------------------------------------------
SEED = 42
TEMPERATURE = 3.0          # must match the prior notebook's evaluate_batch
BATCH_SIZE  = 256
THRESHOLD   = 0.48         # fixed, from the prior calibration-set sweep. No
                           # per-router recalibration in this notebook.

# Ensemble composition: model_9 (guychuk) dropped, model_10 (Aegis) added.
# model_10 lives in the same MODELS_DIRECTORY as model_1..model_8.
# This list's ORDER is the router's class order (index 0..8) and must match
# the order finetune_probs is built in below -- everything downstream is
# positional, not keyed by dataset id, so keep this list as the single
# source of truth for ordering.
MODEL_IDS = [1, 2, 3, 4, 5, 6, 7, 8, 10]
MODEL_DIRS = [MODELS_DIRECTORY + f'model_{i}' for i in MODEL_IDS]
N_MODELS = len(MODEL_IDS)

# Router class index for each dataset id. Cannot just do (dataset_id - 1),
# since id 10 - 1 = 9 would be out of range for a 9-class problem now that
# dataset 9 is missing from the middle of the range.
LABEL_MAP = {ds_id: idx for idx, ds_id in enumerate(MODEL_IDS)}
print('LABEL_MAP:', LABEL_MAP)

MAX_CAL_PER_DATASET = 8000   # cap while iterating; set to None for the final run

USE_DISTILBERT_ROUTER = True
DISTILBERT_EPOCHS = 2
DISTILBERT_MAXLEN = 256

In [ ]:
!hf auth login --token ''

In [ ]:
# router_lab.py must sit somewhere importable
import sys
sys.path.insert(0, BASE_DIRECTORY)
import router_lab as rl

## 1. Load the 8 original training datasets

Same loaders as the earlier router notebook, including the MPDD (`isMalicious`
column) and jayavibhav (integer labels) fixes already worked out there. Dataset
9 (guychuk) is excluded entirely -- not loaded, not in `C_global`, not in the
ensemble.

In [ ]:
def normalise(df, text_col=None):
    """Force a dataframe down to exactly ['prompt', 'label']."""
    if text_col and text_col in df.columns:
        df = df.rename(columns={text_col: 'prompt'})
    if 'prompt' not in df.columns:
        for c in ('text', 'input', 'sentence', 'content', 'question'):
            if c in df.columns:
                df = df.rename(columns={c: 'prompt'})
                break
    if 'prompt' not in df.columns:
        obj = [c for c in df.columns
               if pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_string_dtype(df[c])]
        obj = [c for c in obj if c != 'label']
        if not obj:
            raise KeyError(f'no text column found in {list(df.columns)}')
        widest = max(obj, key=lambda c: df[c].astype(str).str.len().mean())
        print(f'  ! no prompt/text column; falling back to {widest!r}')
        df = df.rename(columns={widest: 'prompt'})
    if 'label' not in df.columns:
        raise KeyError(f'no label column in {list(df.columns)}')
    return df[['prompt', 'label']]


def to_binary_label(series, name=''):
    if series.dtype == bool:
        return series.astype(int)
    if pd.api.types.is_numeric_dtype(series):
        vals = sorted(set(pd.unique(series.dropna())))
        if not set(vals) <= {0, 1}:
            print(f'  . {name}: classes {vals} -> binarised (0 benign, >0 malicious)')
        return (series.fillna(0).astype(float) > 0).astype(int)
    benign = {'benign', 'valid', 'safe', 'legit', 'legitimate', 'clean', 'normal',
              'harmless', 'no', 'false', 'good'}
    malicious = {'malicious', 'jailbreak', 'jailbreaking', 'injection', 'unsafe',
                 'harmful', 'attack', 'toxic', 'yes', 'true', 'bad', 'adversarial'}
    txt = series.astype(str).str.strip().str.lower()
    out = pd.Series(np.nan, index=series.index, dtype='float64')
    out[txt.isin(benign)] = 0
    out[txt.isin(malicious)] = 1
    if out.isna().any():
        raise ValueError(f'{name}: unmapped labels:\n'
                         f'{txt[out.isna()].value_counts().head(10).to_string()}')
    return out.astype(int)


def load_1_synapsecai():
    df = pd.read_parquet(DATASETS_DIRECTORY + "synthetic-prompt-injections_train.parquet")
    return normalise(df, 'text')


def load_2_mpdd():
    df = pd.read_csv(DATASETS_DIRECTORY + "MPDD.csv")
    df.columns = [c.strip().lower() for c in df.columns]
    df = df.rename(columns={'text': 'prompt', 'ismalicious': 'label'})
    df['label'] = to_binary_label(df['label'], 'MPDD')
    return df[['prompt', 'label']]


def load_3_in_the_wild():
    p1 = load_dataset('TrustAIRLab/in-the-wild-jailbreak-prompts',
                      'jailbreak_2023_12_25', split='train').to_pandas()
    p2 = load_dataset('TrustAIRLab/in-the-wild-jailbreak-prompts',
                      'regular_2023_12_25', split='train').to_pandas()
    df = pd.concat([p1, p2], ignore_index=True)[['prompt', 'jailbreak']]
    df['label'] = df['jailbreak'].map({True: 1, False: 0})
    return normalise(df.drop(columns=['jailbreak']))


def load_4_harelix():
    df = pd.read_csv(DATASETS_DIRECTORY + 'harelix_data.csv', names=['prompt', 'label'])
    df['label'] = df['label'].map({'malicious': 1, 'valid': 0})
    return normalise(df)


def load_5_jackhhao():
    df = load_dataset("jackhhao/jailbreak-classification")['train'].to_pandas()
    df['label'] = df['type'].map({'benign': 0, 'jailbreak': 1})
    return normalise(df)


def load_6_qualifire():
    df = load_dataset("qualifire/prompt-injections-benchmark")['test'].to_pandas()
    df = normalise(df, 'text')
    df['label'] = df['label'].map({'benign': 0, 'jailbreak': 1})
    return df


def load_7_jayavibhav():
    df = load_dataset("jayavibhav/prompt-injection-safety")['test'].to_pandas()
    df = df.rename(columns={'text': 'prompt'})
    df['label'] = to_binary_label(df['label'], 'jayavibhav')
    return df[['prompt', 'label']]


def load_8_toxicdetector():
    df = pd.read_csv(DATASETS_DIRECTORY + 'toxic_scenarios.csv')
    df = df[['Question_benign', 'Question_malicious']]
    rng = np.random.RandomState(SEED)
    df['label'] = rng.randint(0, 2, size=len(df))
    df['prompt'] = np.where(df['label'] == 0, df['Question_benign'], df['Question_malicious'])
    df = df[['prompt', 'label']].sample(frac=1, random_state=SEED).reset_index(drop=True)
    return normalise(df)


# (dataset_id, name, loader, stratify) -- dataset 9 (guychuk) intentionally absent
REGISTRY = [
    (1, 'synapsecai/synthetic-prompt-injections', load_1_synapsecai, False),
    (2, 'MPDD',                                   load_2_mpdd,       False),
    (3, 'in-the-wild-jailbreak-prompts',          load_3_in_the_wild, False),
    (4, 'Harelix/Prompt-Injection-Mixed',         load_4_harelix,    False),
    (5, 'jackhhao/jailbreak-classification',      load_5_jackhhao,   False),
    (6, 'qualifire/prompt-injections-benchmark',  load_6_qualifire,  True),
    (7, 'jayavibhav/prompt-injection-safety',     load_7_jayavibhav, True),
    (8, 'ToxicDetector eval',                     load_8_toxicdetector, False),
]

In [ ]:
raw = {}
report = []

for ds_id, name, loader, stratify in REGISTRY:
    try:
        df = loader()
        df = df.dropna(subset=['prompt', 'label'])
        df['prompt'] = df['prompt'].astype(str)
        df['label'] = df['label'].astype(int)
        raw[ds_id] = df
        report.append({'id': ds_id, 'dataset': name, 'rows': len(df),
                       'malicious_frac': round(float(df['label'].mean()), 3),
                       'status': 'ok'})
    except Exception as e:
        report.append({'id': ds_id, 'dataset': name, 'rows': 0,
                       'malicious_frac': None,
                       'status': f'FAILED: {type(e).__name__}: {e}'})

report = pd.DataFrame(report)
missing = report[report['status'] != 'ok']
if len(missing):
    print('!! these datasets did not load:')
    for _, r in missing.iterrows():
        print(f"   {r['id']}. {r['dataset']}: {r['status']}")
else:
    print(f'all {len(REGISTRY)} datasets loaded, {report["rows"].sum():,} rows total')
report

In [ ]:
assert len(raw) == len(REGISTRY), 'fix the loader failures above before continuing'

## 2. Load Aegis and carve its calibration slice

Reuses the prior notebook's exact split (`test_size=0.2, stratify=label, random_state=42`)
so `train_df` here is identical to what `model_10` was fine-tuned on
(`BAGEL_new_finetune.ipynb`). The calibration slice is then carved from that
80%, sized to 10% of Aegis's *total* row count (matching the proportion the
other 8 datasets get), not 10% of the 80% split.

In [ ]:
ds_aegis = load_dataset("nvidia/Aegis-AI-Content-Safety-Dataset-2.0", split="train")
df_aegis = ds_aegis.to_pandas()
df_aegis = df_aegis[["prompt", "prompt_label"]].copy()
df_aegis["label"] = df_aegis["prompt_label"].map({"safe": 0, "unsafe": 1})
df_aegis = df_aegis[["prompt", "label"]].dropna().reset_index(drop=True)
df_aegis["label"] = df_aegis["label"].astype(int)

# Same split as BAGEL_new_finetune.ipynb -- aegis_train is what model_10 was
# actually fine-tuned on. Do not change test_size/random_state here.
aegis_train, aegis_test = train_test_split(
    df_aegis, test_size=0.2, stratify=df_aegis['label'], random_state=42)

n_total = len(df_aegis)
n_cal = int(round(0.10 * n_total))
print(f'Aegis: {n_total:,} total, train={len(aegis_train):,}, test={len(aegis_test):,}, '
     f'carving {n_cal:,} rows (10% of total) for calibration')

aegis_cal = aegis_train.sample(n=n_cal, random_state=SEED)
aegis_cal = aegis_cal[['prompt', 'label']].reset_index(drop=True).assign(model=10)
print(aegis_cal['label'].value_counts())

## 3. Build C_global

8 original datasets' 10% calibration slices (same `split_dataset`/`cap`
convention as before) plus the Aegis slice just carved, all labelled with
their `model` (dataset id) column and then remapped to router class indices
via `LABEL_MAP`.

In [ ]:
def split_dataset(df, stratify):
    strat = df['label'] if stratify else None
    train_df, temp_df = train_test_split(df, test_size=0.30, random_state=SEED, stratify=strat)
    cal_df, test_df = train_test_split(temp_df, test_size=2/3, random_state=SEED)
    return cal_df, test_df


def cap(df, n):
    if n is None or len(df) <= n:
        return df
    return df.sample(n=n, random_state=SEED)


cal_parts, sizes = [], []
for ds_id, name, loader, stratify in REGISTRY:
    df = raw[ds_id]
    cal_df, _test_df = split_dataset(df, stratify)     # test_df unused: test set is the held-out concat, not this
    cal_df = cap(cal_df, MAX_CAL_PER_DATASET).assign(model=ds_id)
    cal_parts.append(cal_df[['prompt', 'label', 'model']])
    sizes.append({'id': ds_id, 'dataset': name, 'total': len(df), 'cal': len(cal_df)})
    print(f'[{ds_id}] {name:42s} total={len(df):>7,}  cal={len(cal_df):>6,}')

cal_parts.append(aegis_cal)
sizes.append({'id': 10, 'dataset': 'nvidia/Aegis (model_10)', 'total': n_total, 'cal': len(aegis_cal)})
print(f'[10] {"nvidia/Aegis (model_10)":42s} total={n_total:>7,}  cal={len(aegis_cal):>6,}')

C_global = pd.concat(cal_parts, ignore_index=True)
C_global['router_class'] = C_global['model'].map(LABEL_MAP)
assert C_global['router_class'].notna().all(), 'a dataset id was not in LABEL_MAP'

C_global.to_csv(ROUTER_DIRECTORY + 'C_global.csv', index=False)
cal_texts = list(C_global['prompt'])
cal_router_labels = C_global['router_class'].values.astype(int)

print(f'\nC_global: {len(C_global):,} rows, {C_global["model"].nunique()} classes')
pd.DataFrame(sizes)

## 4. Train the DistilBERT router

Trained on `C_global`'s prompt text and the remapped router-class labels
(provenance, i.e. which of the 9 promptcops trained on this prompt's source
dataset) -- same target definition as before, just over 9 classes with a
different composition.

In [ ]:
ROUTER_SAVE_PATH = ROUTER_DIRECTORY + 'distilbert_router'

router = rl.DistilBertRouter(max_length=DISTILBERT_MAXLEN, epochs=DISTILBERT_EPOCHS,
                             class_weight='balanced', name='DistilBERT (proposed)')

if os.path.exists(ROUTER_SAVE_PATH):
    router.load(ROUTER_SAVE_PATH)
    print('loaded existing router from', ROUTER_SAVE_PATH)
else:
    router.fit({'texts': cal_texts}, cal_router_labels)
    router.save(ROUTER_SAVE_PATH)
    print('router trained,', router.n_params, 'params, saved to', ROUTER_SAVE_PATH)

## 5. Build the held-out test set

Verbatim from `BAGEL_test_evaluation.ipynb` -- same five datasets, same caps,
same filters, same seed (42) -- so this reproduces the exact rows previously
evaluated against.

In [ ]:
ds_hlyn = load_dataset("hlyn-labs/prompt-injection-judge-deberta-dataset", split="train", token=True)
ds_hlyn = ds_hlyn.shuffle(seed=42).select(range(min(1000, len(ds_hlyn))))
df_hlyn = ds_hlyn.to_pandas()
df_hlyn = df_hlyn[["text", "label"]].rename(columns={"text": "prompt"}).reset_index(drop=True)
df_hlyn["label"] = df_hlyn["label"].astype(int)

In [ ]:
ds_wildjailbreak = load_dataset("allenai/wildjailbreak", "eval", delimiter="\t",
                                keep_default_na=False, split="train")
df_wildjailbreak = ds_wildjailbreak.to_pandas()


def get_wildjailbreak_prompt(row):
    if str(row["data_type"]).startswith("adversarial"):
        return row["adversarial"]
    return row["vanilla"]


df_wildjailbreak["prompt"] = df_wildjailbreak.apply(get_wildjailbreak_prompt, axis=1)

wildjailbreak_label_map = {
    "vanilla_harmful": 1, "adversarial_harmful": 1,
    "vanilla_benign": 0, "adversarial_benign": 0,
}
df_wildjailbreak["label"] = df_wildjailbreak["data_type"].map(wildjailbreak_label_map)
df_wildjailbreak = df_wildjailbreak[["prompt", "label"]].dropna()
df_wildjailbreak["label"] = df_wildjailbreak["label"].astype(int)

In [ ]:
import requests

api_url = (
    "https://api.github.com/repos/tml-epfl/llm-past-tense/"
    "contents/jailbreak_artifacts/past_tense"
)
response = requests.get(api_url)
response.raise_for_status()
files = response.json()

prompts = []
for file_info in files:
    if file_info["name"].endswith(".json"):
        file_response = requests.get(file_info["download_url"])
        file_response.raise_for_status()
        data = file_response.json()
        for artifact in data["jb_artifacts"]:
            prompt = artifact.get("request_reformulated")
            if prompt is not None:
                prompts.append(prompt)

df_past_tense = pd.DataFrame({"prompt": prompts, "label": 1})
df_past_tense = df_past_tense.drop_duplicates(subset="prompt").reset_index(drop=True)
print(df_past_tense.shape)
len(df_past_tense)

In [ ]:
ds_necent = load_dataset("Necent/llm-jailbreak-prompt-injection-dataset", split="train")

training_source_patterns = [
    "synapsecai", "synthetic-prompt-injection",
    "mpdd", "malicious prompt detection",
    "trustairlab", "in-the-wild-jailbreak",
    "harelix", "prompt-injection-mixed-techniques",
    "jackhhao", "jailbreak-classification",
    "qualifire", "prompt-injections-benchmark",
    "jayavibhav", "prompt-injection-safety",
    "toxicdetector",
    "guychuk", "benign-malicious-prompt-classification",
    "aegis"
]
multilingual_source_patterns = [
    "multijail", "ayaredteaming", "aya red teaming",
    "lumees", "multilingual-safety", "francophonia", "linguasafe",
]


def necent_source_allowed(source):
    source = str(source).lower()
    bad_training = any(p in source for p in training_source_patterns)
    bad_multilingual = any(p in source for p in multilingual_source_patterns)
    return not (bad_training or bad_multilingual)


ds_necent = ds_necent.filter(lambda x: necent_source_allowed(x["source"]))
ds_necent = ds_necent.filter(lambda x: str(x["language"]).lower().startswith("en"))
n_necent = min(1000, len(ds_necent))
ds_necent = ds_necent.shuffle(seed=42).select(range(n_necent))

df_necent = ds_necent.to_pandas()
df_necent = df_necent[["prompt", "is_dangerous"]].rename(columns={"is_dangerous": "label"}).reset_index(drop=True)
df_necent["label"] = df_necent["label"].astype(int)

In [ ]:
import os, json
from huggingface_hub import snapshot_download

repo_path = snapshot_download(repo_id="TrustAIRLab/JailbreakQR", repo_type="dataset", token=True)
dataset_dir = os.path.join(repo_path, "dataset_json")

jailbreakqr_prompts = []
for filename in sorted(os.listdir(dataset_dir)):
    if filename.endswith(".json"):
        with open(os.path.join(dataset_dir, filename), "r", encoding="utf-8") as f:
            data = json.load(f)
        for item in data["jailbreaks"]:
            prompt = item.get("prompt")
            if prompt is not None and str(prompt).strip():
                jailbreakqr_prompts.append(prompt)

df_jailbreakqr = pd.DataFrame({"prompt": jailbreakqr_prompts, "label": 1})
len(df_jailbreakqr)

In [ ]:
df_heldout = pd.concat(
    [df_hlyn, df_wildjailbreak, df_past_tense, df_necent, df_jailbreakqr],
    ignore_index=True
)
df_heldout['prompt'] = df_heldout['prompt'].astype(str)
df_heldout['label'] = df_heldout['label'].astype(int)

df_heldout.to_csv(ROUTER_DIRECTORY + 'heldout_test.csv', index=False)

test_texts = df_heldout['prompt'].tolist()
test_labels = df_heldout['label'].tolist()

print("Combined shape:", df_heldout.shape)
print(df_heldout["label"].value_counts())

## 6. Score the 9 promptcops and the router on the held-out set

In [ ]:
finetune_probs_list = rl.score_promptcops(MODEL_DIRS, test_texts,
                                          cache_path=ROUTER_DIRECTORY + 'P_heldout.npy',
                                          batch_size=BATCH_SIZE, temperature=TEMPERATURE)
finetune_probs = [finetune_probs_list[:, j] for j in range(N_MODELS)]   # list-of-arrays, matches the prior notebook's structure
print(finetune_probs_list.shape)

In [ ]:
prompt_injection_model_name = 'meta-llama/Llama-Prompt-Guard-2-86M'
BASELINE_PATH = ROUTER_DIRECTORY + 'baseline_heldout.npy'
if os.path.exists(BASELINE_PATH):
    baseline_probs = np.load(BASELINE_PATH)
else:
    baseline_probs = np.array(rl.score_texts(prompt_injection_model_name, test_texts,
                                             batch_size=BATCH_SIZE, temperature=TEMPERATURE))
    np.save(BASELINE_PATH, baseline_probs)

baseline_preds = [1 if p > 0.5 else 0 for p in baseline_probs]

In [ ]:
def compute_asr_fpr(labels, preds):
    """Identical to the prior notebook's version -- ASR = 1 - (fraction of attacks caught)."""
    labels = [int(x) for x in labels]
    preds = [int(x) for x in preds]
    attack_idx = [i for i, y in enumerate(labels) if y == 1]
    benign_idx = [i for i, y in enumerate(labels) if y == 0]
    asr = sum(preds[i] for i in attack_idx) / len(attack_idx)
    fpr = sum(preds[i] for i in benign_idx) / len(benign_idx)
    return {"ASR": (1 - asr), "FPR": fpr}


baseline_metrics = compute_asr_fpr(test_labels, baseline_preds)
baseline_asr = baseline_metrics['ASR']
baseline_fpr = baseline_metrics['FPR']
print('baseline:', baseline_metrics)

In [ ]:
distilbert_pred_idx = router.predict({'texts': test_texts})   # (N,) ints in 0..N_MODELS-1
print('distilbert predictions:', distilbert_pred_idx.shape)
print('class distribution:', np.bincount(distilbert_pred_idx, minlength=N_MODELS))

## 7. Random Selection curve (matches the prior notebook's mechanism)

Copied from `BAGEL_test_evaluation.ipynb` cell 37 with no changes beyond
variable names already matching -- same `random.seed(42)`, same
`random.sample`, same loop structure, same fixed threshold -- so this
reproduces that curve exactly.

In [ ]:
n_samples = len(test_labels)
random_results = []

random.seed(42)
for n in range(1, N_MODELS + 1):
    combined_probs = []
    for i in range(n_samples):
        selected_indices = random.sample(range(N_MODELS), n)
        avg_prob = np.mean([finetune_probs[j][i] for j in selected_indices])
        combined_probs.append(avg_prob)

    preds = [1 if p > THRESHOLD else 0 for p in combined_probs]
    metrics = compute_asr_fpr(test_labels, preds)
    metrics['n'] = n
    random_results.append(metrics)

random_n = [r['n'] for r in random_results]
random_asr = [r['ASR'] for r in random_results]
random_fpr = [r['FPR'] for r in random_results]

## 8. DistilBERT-routed curve

Same loop structure and same seeding convention (`random.seed(42)` reset
immediately before this loop, mirroring the prior notebook's reset before its own), but
each row's selection is `{distilbert's top pick} u random.sample(remaining
N_MODELS-1, n-1)` instead of `random.sample(all N_MODELS, n)` -- the paper's
mechanism (`S_x = {M_i*} u {M_rand}`), same as used throughout this project.

In [ ]:
distilbert_results = []

random.seed(42)
for n in range(1, N_MODELS + 1):
    combined_probs = []
    for i in range(n_samples):
        top = int(distilbert_pred_idx[i])
        if n == 1:
            selected_indices = [top]
        else:
            remaining = [j for j in range(N_MODELS) if j != top]
            selected_indices = [top] + random.sample(remaining, n - 1)
        avg_prob = np.mean([finetune_probs[j][i] for j in selected_indices])
        combined_probs.append(avg_prob)

    preds = [1 if p > THRESHOLD else 0 for p in combined_probs]
    metrics = compute_asr_fpr(test_labels, preds)
    metrics['n'] = n
    distilbert_results.append(metrics)

distilbert_n = [r['n'] for r in distilbert_results]
distilbert_asr = [r['ASR'] for r in distilbert_results]
distilbert_fpr = [r['FPR'] for r in distilbert_results]

## 9. Final F1 at n = N_MODELS, for both strategies

In [ ]:
def final_f1(results_list, finetune_probs, pred_fn, seed=42):
    random.seed(seed)
    n_final = N_MODELS
    final_probs = []
    for i in range(n_samples):
        selected_indices = pred_fn(i, n_final)
        final_probs.append(np.mean([finetune_probs[j][i] for j in selected_indices]))
    preds = [1 if p > THRESHOLD else 0 for p in final_probs]
    return f1_score(test_labels, preds)


random_f1 = final_f1(random_results, finetune_probs,
                     lambda i, n: random.sample(range(N_MODELS), n))


def distilbert_pick(i, n):
    top = int(distilbert_pred_idx[i])
    if n == 1:
        return [top]
    remaining = [j for j in range(N_MODELS) if j != top]
    return [top] + random.sample(remaining, n - 1)


distilbert_f1 = final_f1(distilbert_results, finetune_probs, distilbert_pick)

print(f'Random Selection  F1={random_f1:.4f}  ASR={random_asr[-1]:.4f}  FPR={random_fpr[-1]:.4f}')
print(f'DistilBERT routed F1={distilbert_f1:.4f}  ASR={distilbert_asr[-1]:.4f}  FPR={distilbert_fpr[-1]:.4f}')

## 10. Plot — extends the prior figure with the DistilBERT curve

In [ ]:
plt.figure(figsize=(9, 6))

plt.axhline(y=baseline_asr, color='C9', alpha=0.5, label='Baseline ASR')
plt.axhline(y=baseline_fpr, color='C3', alpha=0.5, label='Baseline FPR')

plt.plot(random_n, random_asr, marker='o', linestyle=':', color='C0', label='Random Selection ASR')
plt.plot(random_n, random_fpr, marker='s', linestyle=':', color='C1', label='Random Selection FPR')

plt.plot(distilbert_n, distilbert_asr, marker='o', linestyle='-', color='C2', label='DistilBERT ASR')
plt.plot(distilbert_n, distilbert_fpr, marker='s', linestyle='-', color='C4', label='DistilBERT FPR')

textstr = (f'Threshold = {THRESHOLD}\n'
          f'Random F1 = {random_f1:.3f}\n'
          f'DistilBERT F1 = {distilbert_f1:.3f}')
plt.text(0.98, 0.5, textstr, transform=plt.gca().transAxes, fontsize=11,
         verticalalignment='top', horizontalalignment='right',
         bbox=dict(facecolor="white", alpha=1.0))

plt.xlabel('Number of PromptCops Selected (n)', fontsize=16)
plt.ylabel('Metric Value', fontsize=16)
plt.title('(k=9) DistilBERT Router vs Random Selection\n(Held Out Test Set)',
         fontweight='bold', fontsize=16)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tick_params(axis='both', which='major', labelsize=11)
plt.legend(loc='upper right', fontsize=9, facecolor='white')
plt.tight_layout()
plt.savefig(ROUTER_DIRECTORY + 'distilbert_vs_random_heldout.pdf', bbox_inches='tight')
plt.show()

## 11. Export

In [ ]:
pd.DataFrame(random_results).to_csv(ROUTER_DIRECTORY + 'random_selection_results.csv', index=False)
pd.DataFrame(distilbert_results).to_csv(ROUTER_DIRECTORY + 'distilbert_results.csv', index=False)
router.save(ROUTER_DIRECTORY + 'distilbert_router')
print('saved to', ROUTER_DIRECTORY)

# Per-Model Thresholding

In [ ]:
from sklearn.metrics import f1_score

def best_threshold(probs, labels, grid=None):
    """F1-maximizing threshold via simple grid search."""
    if grid is None:
        grid = np.arange(0.05, 0.95, 0.01)
    labels = np.asarray(labels)
    best_f1, best_t = -1, None
    for t in grid:
        preds = (probs > t).astype(int)
        f1 = f1_score(labels, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return round(float(best_t), 2), best_f1

In [ ]:
CAL_THRESH_PATH = ROUTER_DIRECTORY + 'df_cal_thresh.csv'

if os.path.exists(CAL_THRESH_PATH):
    df_cal_thresh = pd.read_csv(CAL_THRESH_PATH)
    print('loaded existing per-model calibration thresholds from', CAL_THRESH_PATH)
else:
    per_model_cal = []
    for idx, ds_id in enumerate(MODEL_IDS):
        own_cal = C_global[C_global['model'] == ds_id]
        own_texts = own_cal['prompt'].tolist()
        own_labels = own_cal['label'].tolist()

        probs = np.array(rl.score_texts(MODEL_DIRS[idx], own_texts,
                                        batch_size=BATCH_SIZE, temperature=TEMPERATURE, fp16=False))
        ideal_t, f1_cal = best_threshold(probs, own_labels)

        per_model_cal.append({'model': f'model_{ds_id}', 'cal_ideal_threshold': ideal_t,
                              'cal_F1': round(f1_cal, 3), 'cal_n': len(own_labels)})
        print(f'model_{ds_id:>2}: calibration-split ideal_threshold={ideal_t:.2f}  '
             f'F1={f1_cal:.3f}  (n={len(own_labels)})')

    df_cal_thresh = pd.DataFrame(per_model_cal)
    df_cal_thresh.to_csv(CAL_THRESH_PATH, index=False)
    print('saved to', CAL_THRESH_PATH)

print('\naverage calibration-split ideal threshold across the 9 models:',
     df_cal_thresh['cal_ideal_threshold'].mean())
df_cal_thresh.sort_values('cal_ideal_threshold')

In [ ]:
# Per-model threshold array, in the same order as MODEL_IDS / finetune_probs
per_model_thresh = df_cal_thresh.set_index('model').loc[
    [f'model_{i}' for i in MODEL_IDS], 'cal_ideal_threshold'
].values

# Shift each promptcop's probability so its own threshold lands at 0.5,
# then use the SAME mean-then-threshold(0.5) rule as before. At n=1 this is
# mathematically identical to comparing the raw probability to that model's
# own threshold -- verified separately, not just assumed.
shifted_probs = [np.clip(0.5 + finetune_probs[j] - per_model_thresh[j], 0, 1)
                 for j in range(N_MODELS)]

In [ ]:
random_results_cal = []
random.seed(42)
for n in range(1, N_MODELS + 1):
    combined_probs = []
    for i in range(n_samples):
        selected_indices = random.sample(range(N_MODELS), n)
        avg_prob = np.mean([shifted_probs[j][i] for j in selected_indices])
        combined_probs.append(avg_prob)
    preds = [1 if p > 0.5 else 0 for p in combined_probs]
    metrics = compute_asr_fpr(test_labels, preds)
    metrics['F1'] = f1_score(test_labels, preds)          # <-- added
    metrics['n'] = n
    random_results_cal.append(metrics)

random_n_cal = [r['n'] for r in random_results_cal]
random_asr_cal = [r['ASR'] for r in random_results_cal]
random_fpr_cal = [r['FPR'] for r in random_results_cal]
random_f1_cal = [r['F1'] for r in random_results_cal]     # <-- added

In [ ]:
distilbert_results_cal = []
random.seed(42)
for n in range(1, N_MODELS + 1):
    combined_probs = []
    for i in range(n_samples):
        top = int(distilbert_pred_idx[i])
        if n == 1:
            selected_indices = [top]
        else:
            remaining = [j for j in range(N_MODELS) if j != top]
            selected_indices = [top] + random.sample(remaining, n - 1)
        avg_prob = np.mean([shifted_probs[j][i] for j in selected_indices])
        combined_probs.append(avg_prob)
    preds = [1 if p > 0.5 else 0 for p in combined_probs]
    metrics = compute_asr_fpr(test_labels, preds)
    metrics['F1'] = f1_score(test_labels, preds)          # <-- added
    metrics['n'] = n
    distilbert_results_cal.append(metrics)

distilbert_n_cal = [r['n'] for r in distilbert_results_cal]
distilbert_asr_cal = [r['ASR'] for r in distilbert_results_cal]
distilbert_fpr_cal = [r['FPR'] for r in distilbert_results_cal]
distilbert_f1_cal = [r['F1'] for r in distilbert_results_cal]   # <-- added

In [ ]:
results_table = pd.DataFrame({
    'n': random_n_cal,
    'Random ASR': random_asr_cal, 'Random FPR': random_fpr_cal, 'Random F1': random_f1_cal,
    'DistilBERT ASR': distilbert_asr_cal, 'DistilBERT FPR': distilbert_fpr_cal, 'DistilBERT F1': distilbert_f1_cal,
})
print(results_table.to_string(index=False))
results_table.to_csv(ROUTER_DIRECTORY + 'per_model_cal_results_table.csv', index=False)

# Same plot, both threshold regimes overlaid for direct comparison
plt.figure(figsize=(10, 7))

plt.axhline(y=baseline_asr, color='C9', alpha=0.4, label='Untrained Llama Promptguard 2 ASR')
plt.axhline(y=baseline_fpr, color='C3', alpha=0.4, label='Untrained Llama Promptguard 2 FPR')

plt.plot(random_n_cal, random_asr_cal, marker='o', linestyle=':', color='C0', label='Random Selection ASR')
plt.plot(random_n_cal, random_fpr_cal, marker='s', linestyle=':', color='C1', label='Random Selection FPR')
plt.plot(distilbert_n_cal, distilbert_asr_cal, marker='o', linestyle='-', color='C2', label='DistilBERT ASR')
plt.plot(distilbert_n_cal, distilbert_fpr_cal, marker='s', linestyle='-', color='C4', label='DistilBERT FPR')

plt.xlabel('Number of PromptCops Selected (n)', fontsize=18)
plt.ylabel('Metric Value', fontsize=18)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=13, ncol=2, loc='upper right')
plt.tight_layout()
plt.savefig(ROUTER_DIRECTORY + 'distilbert_vs_random_percal.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ---- Random Selection only, per-model calibrated threshold ----
plt.figure(figsize=(9, 6))

plt.axhline(y=baseline_asr, color='C9', alpha=0.4, label='Untrained Llama Promptguard 2 ASR')
plt.axhline(y=baseline_fpr, color='C3', alpha=0.4, label='Untrained Llama Promptguard 2 FPR')

plt.plot(random_n_cal, random_asr_cal, marker='o', linestyle=':', color='C0', label='Random Selection ASR')
plt.plot(random_n_cal, random_fpr_cal, marker='s', linestyle=':', color='C1', label='Random Selection FPR')

final_random_f1 = random_f1_cal[-1]   # n = N_MODELS, full ensemble
textstr = f'Random F1 = {final_random_f1:.3f}'
plt.text(0.98, 0.5, textstr, transform=plt.gca().transAxes, fontsize=11,
         verticalalignment='top', horizontalalignment='right',
         bbox=dict(facecolor='white', alpha=1.0))

plt.xlabel('Number of PromptCops Selected (n)', fontsize=14)
plt.ylabel('Metric Value', fontsize=14)
plt.title('Bagel Performance on Held Out Test Set',
         fontweight='bold', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=9, loc='upper right')
plt.tight_layout()
plt.savefig(ROUTER_DIRECTORY + 'random_only_percal.pdf', bbox_inches='tight')
plt.show()

# Performance Over Time


In [ ]:
# ---- Isolate the WildJailbreak-eval slice (no rescoring for models 1-8,10) ----
wj_start = len(df_hlyn)
wj_end = wj_start + len(df_wildjailbreak)
assert df_heldout['prompt'].iloc[wj_start:wj_end].tolist() == df_wildjailbreak['prompt'].tolist(), \
    "index slice doesn't line up with df_wildjailbreak -- did the df_heldout concat order change?"

wj_texts = test_texts[wj_start:wj_end]
wj_labels = test_labels[wj_start:wj_end]
wj_finetune_probs_before = [p[wj_start:wj_end] for p in finetune_probs]   # 9 models, WildJailbreak-eval only

print(f'WildJailbreak-eval slice: {len(wj_texts)} rows (expected {len(df_wildjailbreak)})')

In [ ]:
# ---- Score model_11 on the WildJailbreak-eval slice ----
MODEL_11_DIR = '/content/drive/MyDrive/Bagel Router Exp/model_11'

model_11_probs_wj = np.array(rl.score_texts(MODEL_11_DIR, wj_texts,
                                            batch_size=BATCH_SIZE, temperature=TEMPERATURE))
wj_finetune_probs_after = wj_finetune_probs_before + [model_11_probs_wj]
print('model_11 scored on WildJailbreak-eval:', model_11_probs_wj.shape)

In [ ]:
# ---- Technique 1: shared threshold (0.48, fixed for both, not recalculated) ----
def random_selection_curve(probs_list, labels, threshold, seed=42):
    n_models = len(probs_list); n_samples = len(labels)
    random.seed(seed)
    results = []
    for n in range(1, n_models + 1):
        combined = []
        for i in range(n_samples):
            idx = random.sample(range(n_models), n)
            combined.append(np.mean([probs_list[j][i] for j in idx]))
        preds = [1 if p > threshold else 0 for p in combined]
        m = compute_asr_fpr(labels, preds)
        m['F1'] = f1_score(labels, preds)
        m['n'] = n
        results.append(m)
    return results

t1_before = random_selection_curve(wj_finetune_probs_before, wj_labels, 0.48)
t1_after  = random_selection_curve(wj_finetune_probs_after,  wj_labels, 0.48)
print('Technique 1 BEFORE (k=9), n=9 :', t1_before[-1])
print('Technique 1 AFTER  (k=10), n=10:', t1_after[-1])

In [ ]:
# ---- Technique 2: model_11's own calibration slice (10% of WildJailbreak-train's
# total, carved from the unused remainder -- zero overlap with the 20,000
# rows model_11 was trained on) ----
ds_wj_train_full = load_dataset("allenai/wildjailbreak", "train", delimiter="\t",
                                keep_default_na=False, split="train")
df_wj_train_full = ds_wj_train_full.to_pandas()

def get_wj_prompt(row):
    return row['adversarial'] if str(row['data_type']).startswith('adversarial') else row['vanilla']

df_wj_train_full['prompt'] = df_wj_train_full.apply(get_wj_prompt, axis=1)
wj_label_map = {'vanilla_harmful': 1, 'adversarial_harmful': 1, 'vanilla_benign': 0, 'adversarial_benign': 0}
df_wj_train_full['label'] = df_wj_train_full['data_type'].map(wj_label_map)
df_wj_train_full = df_wj_train_full[['prompt', 'label']].dropna().reset_index(drop=True)
df_wj_train_full['label'] = df_wj_train_full['label'].astype(int)

MODEL_11_TRAIN_SEED = 42       # must match train_model_11.ipynb's SEED
MODEL_11_TRAIN_SIZE = 20_000   # must match SUBSAMPLE_SIZE actually used

used_idx = df_wj_train_full.sample(n=MODEL_11_TRAIN_SIZE, random_state=MODEL_11_TRAIN_SEED).index
unused_df = df_wj_train_full.drop(index=used_idx).reset_index(drop=True)

n_cal_11 = int(round(0.10 * len(df_wj_train_full)))   # same 10%-of-total convention as model_10/Aegis
model_11_cal = unused_df.sample(n=n_cal_11, random_state=SEED)
print(f'model_11 calibration slice: {len(model_11_cal)} rows, zero overlap with the '
     f'{MODEL_11_TRAIN_SIZE:,} training rows (by construction)')

In [ ]:
model_11_cal_texts = model_11_cal['prompt'].tolist()
model_11_cal_labels = model_11_cal['label'].tolist()

model_11_cal_probs = np.array(rl.score_texts(MODEL_11_DIR, model_11_cal_texts,
                                             batch_size=BATCH_SIZE, temperature=TEMPERATURE))
model_11_ideal_t, model_11_cal_f1 = best_threshold(model_11_cal_probs, model_11_cal_labels)
print(f'model_11 own calibration-split ideal threshold: {model_11_ideal_t:.2f}  '
     f'F1={model_11_cal_f1:.3f}  (n={len(model_11_cal)})')

per_model_thresh_after = np.append(per_model_thresh, model_11_ideal_t)

In [ ]:
# ---- Technique 2: shift-and-average, extended to k=10 ----
wj_shifted_probs_before = [p[wj_start:wj_end] for p in shifted_probs]   # reuse, already computed

wj_shifted_probs_after = [
    np.clip(0.5 + wj_finetune_probs_after[j] - per_model_thresh_after[j], 0, 1)
    for j in range(len(wj_finetune_probs_after))
]

t2_before = random_selection_curve(wj_shifted_probs_before, wj_labels, 0.5)
t2_after  = random_selection_curve(wj_shifted_probs_after,  wj_labels, 0.5)
print('Technique 2 BEFORE (k=9), n=9 :', t2_before[-1])
print('Technique 2 AFTER  (k=10), n=10:', t2_after[-1])

In [ ]:
# ---- Comparison table + plot, both techniques ----
comparison_wj = pd.DataFrame([
    {'technique': 'Shared (0.48)',     'k': 9,  **{m: t1_before[-1][m] for m in ('ASR','FPR','F1')}},
    {'technique': 'Shared (0.48)',     'k': 10, **{m: t1_after[-1][m]  for m in ('ASR','FPR','F1')}},
    {'technique': 'Per-model cal',     'k': 9,  **{m: t2_before[-1][m] for m in ('ASR','FPR','F1')}},
    {'technique': 'Per-model cal',     'k': 10, **{m: t2_after[-1][m]  for m in ('ASR','FPR','F1')}},
])
print(comparison_wj.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
for ax, (before, after, title) in zip(
    axes,
    [(t1_before, t1_after, 'Shared Threshold (0.48)'), (t2_before, t2_after, 'Per-Model Calibrated')]
):
    ax.plot([r['n'] for r in before], [r['F1'] for r in before], marker='o', linestyle='--', label='F1, k=9 (before)')
    ax.plot([r['n'] for r in after],  [r['F1'] for r in after],  marker='o', linestyle='-',  label='F1, k=10 (after model_11)')
    ax.set_xlabel('Number of PromptCops Selected (n)')
    ax.set_title(title)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend()
axes[0].set_ylabel('F1')
plt.suptitle('model_11 Before/After — WildJailbreak-eval only', fontweight='bold')
plt.tight_layout()
plt.savefig(ROUTER_DIRECTORY + 'model11_before_after_wildjailbreak.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ---- ASR/FPR comparison plot, both techniques (separate from the F1 plot) ----
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
for ax, (before, after, title) in zip(
    axes,
    [(t1_before, t1_after, 'Shared Threshold (0.48)'), (t2_before, t2_after, 'Per-Model Calibrated')]
):
    ax.plot([r['n'] for r in before], [r['ASR'] for r in before], marker='o', linestyle='--', color='C0', label='ASR, k=9 (before)')
    ax.plot([r['n'] for r in after],  [r['ASR'] for r in after],  marker='o', linestyle='-',  color='C0', label='ASR, k=10 (after model_11)')
    ax.plot([r['n'] for r in before], [r['FPR'] for r in before], marker='s', linestyle='--', color='C1', label='FPR, k=9 (before)')
    ax.plot([r['n'] for r in after],  [r['FPR'] for r in after],  marker='s', linestyle='-',  color='C1', label='FPR, k=10 (after model_11)')
    ax.set_xlabel('Number of PromptCops Selected (n)')
    ax.set_title(title)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(fontsize=8)
axes[0].set_ylabel('Metric Value')
plt.suptitle('model_11 Before/After — ASR & FPR (WildJailbreak-eval only)', fontweight='bold')
plt.tight_layout()
plt.savefig(ROUTER_DIRECTORY + 'model11_before_after_asr_fpr.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ---- Extend router training data to include model_11 (reuses model_11_cal_texts/
# model_11_cal_labels already built for Technique 2 -- no new calibration slice needed) ----
cal_texts_after = cal_texts + model_11_cal_texts
cal_router_labels_after = np.concatenate([cal_router_labels, np.full(len(model_11_cal_texts), 9)])
print(f'extended router training set: {len(cal_texts_after):,} rows, '
     f'{len(np.unique(cal_router_labels_after))} classes')

In [ ]:
# ---- Train (or load) the k=10 router -- SEPARATE folder, does not touch the
# original k=9 router saved earlier ----
ROUTER_K10_SAVE_PATH = ROUTER_DIRECTORY + 'distilbert_router_k10'

router_k10 = rl.DistilBertRouter(max_length=DISTILBERT_MAXLEN, epochs=DISTILBERT_EPOCHS,
                                 class_weight='balanced', name='DistilBERT (k=10, proposed)')

if os.path.exists(ROUTER_K10_SAVE_PATH):
    router_k10.load(ROUTER_K10_SAVE_PATH)
    print('loaded existing k=10 router from', ROUTER_K10_SAVE_PATH)
else:
    router_k10.fit({'texts': cal_texts_after}, cal_router_labels_after)
    router_k10.save(ROUTER_K10_SAVE_PATH)
    print('k=10 router trained,', router_k10.n_params, 'params, saved to', ROUTER_K10_SAVE_PATH)

In [ ]:
# ---- Routing decisions on the WildJailbreak-eval slice ----
# BEFORE: slice the ALREADY-COMPUTED k=9 router predictions -- no retraining, no rescoring
distilbert_pred_idx_before_wj = distilbert_pred_idx[wj_start:wj_end]
# AFTER: the new k=10 router, run fresh on the same wj_texts
distilbert_pred_idx_after_wj = router_k10.predict({'texts': wj_texts})

print('BEFORE (k=9) routing distribution: ', np.bincount(distilbert_pred_idx_before_wj, minlength=9))
print('AFTER  (k=10) routing distribution:', np.bincount(distilbert_pred_idx_after_wj, minlength=10))

In [ ]:
# ---- Generic routed curve: top pick (from the router) + n-1 random peers ----
def distilbert_routed_curve(probs_list, labels, pred_idx, threshold, seed=42):
    n_models = len(probs_list); n_samples = len(labels)
    random.seed(seed)
    results = []
    for n in range(1, n_models + 1):
        combined = []
        for i in range(n_samples):
            top = int(pred_idx[i])
            if n == 1:
                idx = [top]
            else:
                remaining = [j for j in range(n_models) if j != top]
                idx = [top] + random.sample(remaining, n - 1)
            combined.append(np.mean([probs_list[j][i] for j in idx]))
        preds = [1 if p > threshold else 0 for p in combined]
        m = compute_asr_fpr(labels, preds)
        m['F1'] = f1_score(labels, preds)
        m['n'] = n
        results.append(m)
    return results

# Technique 1 (shared 0.48)
d1_before = distilbert_routed_curve(wj_finetune_probs_before, wj_labels, distilbert_pred_idx_before_wj, 0.48)
d1_after  = distilbert_routed_curve(wj_finetune_probs_after,  wj_labels, distilbert_pred_idx_after_wj,  0.48)

# Technique 2 (per-model calibrated, threshold=0.5 on shifted probs)
d2_before = distilbert_routed_curve(wj_shifted_probs_before, wj_labels, distilbert_pred_idx_before_wj, 0.5)
d2_after  = distilbert_routed_curve(wj_shifted_probs_after,  wj_labels, distilbert_pred_idx_after_wj,  0.5)

print('DistilBERT-routed, Technique 1, BEFORE n=9 :', d1_before[-1])
print('DistilBERT-routed, Technique 1, AFTER  n=10:', d1_after[-1])
print('DistilBERT-routed, Technique 2, BEFORE n=9 :', d2_before[-1])
print('DistilBERT-routed, Technique 2, AFTER  n=10:', d2_after[-1])

In [ ]:
# ---- Extended comparison table + F1 plot: Random Selection vs DistilBERT-routed ----
comparison_wj_full = pd.DataFrame([
    {'strategy': 'Random',     'technique': 'Shared (0.48)', 'k': 9,  **{m: t1_before[-1][m] for m in ('ASR','FPR','F1')}},
    {'strategy': 'Random',     'technique': 'Shared (0.48)', 'k': 10, **{m: t1_after[-1][m]  for m in ('ASR','FPR','F1')}},
    {'strategy': 'DistilBERT', 'technique': 'Shared (0.48)', 'k': 9,  **{m: d1_before[-1][m] for m in ('ASR','FPR','F1')}},
    {'strategy': 'DistilBERT', 'technique': 'Shared (0.48)', 'k': 10, **{m: d1_after[-1][m]  for m in ('ASR','FPR','F1')}},
    {'strategy': 'Random',     'technique': 'Per-model cal', 'k': 9,  **{m: t2_before[-1][m] for m in ('ASR','FPR','F1')}},
    {'strategy': 'Random',     'technique': 'Per-model cal', 'k': 10, **{m: t2_after[-1][m]  for m in ('ASR','FPR','F1')}},
    {'strategy': 'DistilBERT', 'technique': 'Per-model cal', 'k': 9,  **{m: d2_before[-1][m] for m in ('ASR','FPR','F1')}},
    {'strategy': 'DistilBERT', 'technique': 'Per-model cal', 'k': 10, **{m: d2_after[-1][m]  for m in ('ASR','FPR','F1')}},
])
print(comparison_wj_full.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
for ax, (r_before, r_after, d_before, d_after, title) in zip(
    axes,
    [(t1_before, t1_after, d1_before, d1_after, 'Shared Threshold (0.48)'),
     (t2_before, t2_after, d2_before, d2_after, 'Per-Model Calibrated')]
):
    ax.plot([r['n'] for r in r_before], [r['F1'] for r in r_before], marker='o', linestyle=':',  color='C0', label='Random, k=9 (before)')
    ax.plot([r['n'] for r in r_after],  [r['F1'] for r in r_after],  marker='o', linestyle='-',  color='C0', label='Random, k=10 (after)')
    ax.plot([r['n'] for r in d_before], [r['F1'] for r in d_before], marker='s', linestyle=':',  color='C2', label='DistilBERT, k=9 (before)')
    ax.plot([r['n'] for r in d_after],  [r['F1'] for r in d_after],  marker='s', linestyle='-',  color='C2', label='DistilBERT, k=10 (after)')
    ax.set_xlabel('Number of PromptCops Selected (n)')
    ax.set_title(title)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(fontsize=8)
axes[0].set_ylabel('F1')
plt.suptitle('model_11 Before/After — Random vs DistilBERT-routed (WildJailbreak-eval only)', fontweight='bold')
plt.tight_layout()
plt.savefig(ROUTER_DIRECTORY + 'model11_before_after_with_routing.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ---- Extended comparison table + F1 plot: Random Selection vs DistilBERT-routed ----
comparison_wj_full = pd.DataFrame([
    {'strategy': 'Random Selection',     'technique': 'Shared (0.48)', 'k': 9,  **{m: t1_before[-1][m] for m in ('ASR','FPR','F1')}},
    {'strategy': 'Random Selection',     'technique': 'Shared (0.48)', 'k': 10, **{m: t1_after[-1][m]  for m in ('ASR','FPR','F1')}},
    {'strategy': 'DistilBERT', 'technique': 'Shared (0.48)', 'k': 9,  **{m: d1_before[-1][m] for m in ('ASR','FPR','F1')}},
    {'strategy': 'DistilBERT', 'technique': 'Shared (0.48)', 'k': 10, **{m: d1_after[-1][m]  for m in ('ASR','FPR','F1')}},
    {'strategy': 'Random Selection',     'technique': 'Per-model cal', 'k': 9,  **{m: t2_before[-1][m] for m in ('ASR','FPR','F1')}},
    {'strategy': 'Random Selection',     'technique': 'Per-model cal', 'k': 10, **{m: t2_after[-1][m]  for m in ('ASR','FPR','F1')}},
    {'strategy': 'DistilBERT', 'technique': 'Per-model cal', 'k': 9,  **{m: d2_before[-1][m] for m in ('ASR','FPR','F1')}},
    {'strategy': 'DistilBERT', 'technique': 'Per-model cal', 'k': 10, **{m: d2_after[-1][m]  for m in ('ASR','FPR','F1')}},
])
print(comparison_wj_full.to_string(index=False))

# Create a single plot instead of 1x2 subplots
fig, ax = plt.subplots(figsize=(8, 6))

# Plot only the "Per-Model Calibrated" data (previously the right subplot)
ax.plot([r['n'] for r in t2_before], [r['F1'] for r in t2_before], marker='o', linestyle=':',  color='C0', label='Random, k=9 (before)')
ax.plot([r['n'] for r in t2_after],  [r['F1'] for r in t2_after],  marker='o', linestyle='-',  color='C0', label='Random, k=10 (after)')
ax.plot([r['n'] for r in d2_before], [r['F1'] for r in d2_before], marker='s', linestyle=':',  color='C2', label='DistilBERT, k=9 (before)')
ax.plot([r['n'] for r in d2_after],  [r['F1'] for r in d2_after],  marker='s', linestyle='-',  color='C2', label='DistilBERT, k=10 (after)')

ax.set_xlabel('Number of PromptCops Selected (n)', fontsize=18)
ax.set_ylabel('F1', fontsize=18)
ax.tick_params(axis='both', labelsize=14)
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend(fontsize=13)

plt.tight_layout()
plt.savefig(ROUTER_DIRECTORY + 'model11_before_after_with_routing.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# from google.colab import runtime
# runtime.unassign()